In [ ]:
pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install --upgrade accelerate
pip install transformers accelerate

In [ ]:
pip install datasets

In [ ]:
pip install evaluate

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
import evaluate

nltk.download("punkt")


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model_1 = "knkarthick/MEETING_SUMMARY"
tokenizer = AutoTokenizer.from_pretrained(model_1)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_1).to(device)

In [ ]:
ds = load_dataset("knkarthick/samsum")

In [ ]:
ds

In [ ]:
save_path = "/content/drive/MyDrive/samsum_dataset"
ds.save_to_disk(save_path)
print(f"✅ Full dataset saved at {save_path}")

In [ ]:
def convert_examples_to_features(batch):
    model_inputs = tokenizer(
        batch["dialogue"],
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["summary"],
            max_length=128,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_ds = ds.map(convert_examples_to_features, batched=True, remove_columns=["id", "dialogue", "summary"])


In [ ]:

def debug_fn(batch):
    print(type(batch))         
    print(batch.keys())        
    print(batch["dialogue"][:2])  
    print(batch["summary"][:2])   
    return batch

ds.map(debug_fn, batched=True, batch_size=2)

In [ ]:
def convert_examples_to_features(batch):
    dialogues = [str(x) for x in batch["dialogue"]]
    summaries = [str(x) for x in batch["summary"]]

    model_inputs = tokenizer(
        dialogues,
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            summaries,
            max_length=128,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [ ]:
tokenized_ds = ds.map(
    convert_examples_to_features,
    batched=True,
)

In [ ]:
tokenized_ds["train"]

In [ ]:
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)


In [ ]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir="trained-samsum",
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01, logging_steps=10,
    eval_steps=500, save_steps=1e6,
    gradient_accumulation_steps=16
)

In [ ]:
trainer = Trainer(model=model_pegasus, args=trainer_args,
                  processing_class=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=tokenized_ds["train"].select(range(5000)),
                  eval_dataset=tokenized_ds["validation"])

In [ ]:
trainer.train()

In [ ]:
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=16, device=device,
                                column_text="article",
                                column_summary="highlights"):
  article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
  target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

  for article_batch, target_batch in tqdm(
      zip(article_batches, target_batches), total=len(article_batches)):
      inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                         padding = "max_length", return_tensors="pt")
      summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                 attention_mask=inputs["attention_mask"].to(device),
                                 length_penalty=0.8, num_beams=8, max_length=128)
      decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                            clean_up_tokenization_spaces=True)
                                 for s in summaries]

      metric.add_batch(predictions=decoded_summaries, references=target_batch)

  score = metric.compute()
  return score



In [ ]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = evaluate.load('rouge')

In [ ]:
score = calculate_metric_on_test_ds(
    ds["test"][0:100], rouge_metric, trainer.model,
    tokenizer, batch_size=2, column_text="dialogue",
    column_summary="summary"
)

rouge_dict = {rn: score[rn] for rn in rouge_names}
pd.DataFrame(rouge_dict, index = [f'pegasus'])

In [ ]:

save_path = "/content/drive/MyDrive/trained-samsum-model"
model_pegasus.save_pretrained(save_path)

In [ ]:
save_directory = "/content/drive/MyDrive/tokenizer"
tokenizer.save_pretrained(save_directory)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/tokenizer")

In [ ]:
gen_kwargs = {"length_penalty":0.8, "num_beams":8, "max_length": 256}

sample_text = ds["test"][0]["dialogue"]
reference = ds["test"][0]["summary"]

pipe = pipeline("summarization", model =model_pegasus, tokenizer = tokenizer)

print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference)

print("\nModel Summary:")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])